# Implementing an Earley Parser

The file `simple.g` contains a grammar describing arithmetic expressions that only use the operators `+` and `*`.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
printfn "%s" (System.IO.File.ReadAllText "simple.g")

In [ ]:
#r "GrammarSyntax/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "GrammarSyntax/bin/Debug/net9.0/GrammarSyntax.dll"
open GrammarSyntax

The scanner and grammar productions are real FsLexYacc specifications. They are built by `dotnet build GrammarSyntax/GrammarSyntax.fsproj` before starting this notebook.

```fsharp
{
open GrammarSyntax.Parser
open FSharp.Text.Lexing
let lexeme (buffer: LexBuffer<char>) = System.String(buffer.Lexeme)
}
let lower = ['a'-'z']
let upper = ['A'-'Z']
let digit = ['0'-'9']
rule token = parse
 | [' ' '\t' '\r']+ { token lexbuf }
 | '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
 | '#' [^'\n']* { token lexbuf }
 | "//" [^'\n']* { token lexbuf }
 | lower (lower | digit | '_')* { VARIABLE(lexeme lexbuf) }
 | upper (upper | digit | '_')* { TOKEN(lexeme lexbuf) }
 | '\'' [^'\'' '\n']* '\'' { let text = lexeme lexbuf in LITERAL(text.Substring(1, text.Length - 2)) }
 | ':' { COLON } | '|' { BAR } | ';' { SEMICOLON }
 | eof { EOF }
 | _ { failwithf "Illegal grammar character at line %d, column %d" lexbuf.StartPos.Line (lexbuf.StartPos.Column + 1) }

```

```fsharp
%{
open GrammarSyntax
%}
%token <string> VARIABLE TOKEN LITERAL
%token COLON BAR SEMICOLON EOF
%start grammar
%type <GrammarSyntax.Rule list> grammar
%%
grammar: rules EOF { $1 }
rules: rule { $1 } | rules rule { $1 @ $2 }
rule: VARIABLE COLON alternatives SEMICOLON { List.map (fun body -> { Head = $1; Body = body }) $3 }
alternatives: body { [$1] } | alternatives BAR body { $1 @ [$3] }
body: { [] } | body symbol { $1 @ [$2] }
symbol: VARIABLE { Nonterminal $1 } | TOKEN { Terminal $1 } | LITERAL { Terminal $1 }

```

A literal is any character that is not the newline character and is enclosed in quotes.
The quote characters are removed.

## Implementing the Parser

---

Below is the grammar that FsLexYacc uses to parse a context free grammar.

---
```
    grammar
        : rule
        | rule grammar
        ;

    rule
        : VARIABLE ':' item_list ';'
        ;

    item_list
        : λ
        | item item_list
        ;

    item
        : VARIABLE
        | TERMINAL
        | LITERAL
        ;
```

A grammar is represented as a list of grammar rules.

The parser creates a typed list of `Rule` records.

A grammar rule is represented as a list of items.

Items are strings.

In [ ]:
let parse = Parse.file

In [ ]:
let grammar = parse "simple.g"
printfn "%A" grammar

## Earley's Algorithm

Given a context-free grammar $G = \langle V, \Sigma, R, s \rangle$ and a string $w = x_1x_2 \cdots x_n \in \Sigma^*$ of length $n$, 
an *Earley item* is a pair of the form
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
such that 
- $(a \rightarrow \alpha \beta) \in R\quad$  and
- $k \in \{0,1,\cdots,n\}$. 

The record stores the production in `Rule`, the length of $\alpha$ in `Dot`, and $k$ in `Origin`. Structural equality and comparison automatically support chart sets.

F# records are immutable and structurally comparable, so `Set<EarleyItem>` automatically deduplicates chart entries.

In [ ]:
type EarleyItem = { Rule: Rule; Dot: int; Origin: int }
type Token = { Kind: string; Lexeme: string }

let isComplete (item: EarleyItem) = item.Dot = item.Rule.Body.Length
let symbolAfterDot (item: EarleyItem) = List.tryItem item.Dot item.Rule.Body
let sameVar name item = symbolAfterDot item = Some(Nonterminal name)
let scanToken token item = symbolAfterDot item = Some(Terminal token.Kind)
let nextVar item =
    match symbolAfterDot item with Some(Nonterminal name) -> Some name | _ -> None
let moveDot (item: EarleyItem) = { item with Dot = item.Dot + 1 }
let startRule grammar = { Head = "ŝ"; Body = [Nonterminal grammar.Start] }
let startItem grammar = { Rule = startRule grammar; Dot = 0; Origin = 0 }
let finishItem grammar = { Rule = startRule grammar; Dot = 1; Origin = 0 }

Given an Earley item `item`, the function `isComplete` checks, whether the Earley item `item` has the form
$$\langle A \rightarrow \alpha \bullet, k \rangle,$$
i.e. whether the $\bullet$ is at the end of the grammar rule.

The function `sameVar c item` checks, whether the item following the dot is the same as the variable 
given as argument, i.e. `sameVar c item` returns `true` if `item` is an Earley item of the form
$$\langle a \rightarrow \alpha \bullet c\beta, k \rangle.$$

The function `scanToken token item` checks, whether the item following the dot matches the token `T`, 
i.e. `scanToken token item` returns `true` if `item` is an Earley item of the form
$$\langle a \rightarrow \alpha \bullet T\beta, k \rangle.$$
The argument $T$ can either be the name of a token or a literal.

Given an Earley item, this function returns the name of the variable following the dot.  If there is no variable following the dot, the function returns `None`.  The `Nonterminal` union case makes this distinction explicit.

The function `moveDot item` moves the $\bullet$ in the Earley item `item`, where `item` has the form 
$$\langle a \rightarrow \alpha \bullet \beta, k \rangle$$
over the next variable, token, or literal in $\beta$.  It assumes that $\beta$ is not empty.

The `Grammar` record stores a start variable and the list of rules. Nonterminals and terminals are separate union cases, including quoted lower-case literals.

The function `startItem` returns the Earley item
$$ \langle\hat{s} \rightarrow \bullet s, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is the fresh augmented variable `ŝ`.

The function `finishItem` returns the Earley item
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle $$
where $s$ is the start variable of the given grammar and $\hat{s}$ is a new variable.

The `Grammar.Start` field records the head of the first production as the start variable.

The parser stores one immutable set per input position in an array. Fixed-point iteration closes a chart position under prediction and completion before scanning.

The method `parse` implements Earley's algorithm.  For all states 
$Q_1$, $\cdots$, $Q_n$ we proceed as follows:
- We apply the *completion* operation followed by the *prediction* operation.
  This is done until no more states are added to $Q_i$.  
  
  (The inner `while` loop is not necessary if the grammar does not contain $\varepsilon$-rules.)
- Finally, the *scanning* operation is applied to $Q_i$.  This operation adds
  items to the set $Q_{i+1}$.

After $Q_i$ has been computed, we proceed to process $Q_{i+1}$.
Parsing is successful iff
$$ \langle\hat{s} \rightarrow s \bullet, 0\rangle \in Q_n $$

In [ ]:
// Prediction and completion only enlarge a chart set.
let predict grammar (chart: Set<EarleyItem> array) position =
    let additions =
        chart[position]
        |> Seq.choose nextVar
        |> Seq.collect (fun name ->
            grammar.Rules
            |> Seq.filter (fun rule -> rule.Head = name)
            |> Seq.map (fun rule -> { Rule = rule; Dot = 0; Origin = position }))
        |> Set.ofSeq
    chart[position] <- Set.union chart[position] additions

let complete (chart: Set<EarleyItem> array) position =
    let additions =
        chart[position]
        |> Seq.filter isComplete
        |> Seq.collect (fun item ->
            chart[item.Origin]
            |> Seq.filter (sameVar item.Rule.Head)
            |> Seq.map moveDot)
        |> Set.ofSeq
    chart[position] <- Set.union chart[position] additions

let scan (tokens: Token array) (chart: Set<EarleyItem> array) position =
    if position < tokens.Length then
        chart[position + 1] <-
            chart[position]
            |> Seq.filter (scanToken tokens[position])
            |> Seq.map moveDot
            |> Set.ofSeq

let earleyParse grammar tokens =
    let input = List.toArray tokens
    let chart = Array.create (input.Length + 1) Set.empty
    chart[0] <- Set.singleton (startItem grammar)
    for position in 0 .. input.Length do
        let mutable changed = true
        while changed do
            let previous = chart[position]
            predict grammar chart position
            complete chart position
            changed <- chart[position] <> previous
        scan input chart position
    chart

let recognizes grammar tokens =
    let chart = earleyParse grammar tokens
    Set.contains (finishItem grammar) chart[tokens.Length]

The method `complete chart i` applies the completion operation to the state $Q_i$:
If we have
 - $\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i$  and
 - $\langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j$,
then the parser tried to parse the variable $c$ after having read `the first j input tokens`
and we know that 
$$ c \Rightarrow^* \texttt{input tokens from j to i, with i excluded}, $$
i.e. the parser has recognized $c$ after having read `input tokens from j to i, with i excluded`.
Therefore the parser should proceed to recognize $\delta$ in state $Q_i$.
Therefore we add the *Earley item* $\langle a \rightarrow \beta c \bullet \delta,k\rangle$ to the set $Q_i$:
$$\langle c \rightarrow \gamma \bullet, j\rangle \in Q_i \wedge
  \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_j \;\rightarrow\;
          Q_i := Q_i \cup \bigl\{ \langle a \rightarrow \beta c \bullet \delta, k\rangle \bigr\}
$$

The method `predict grammar chart i` applies the prediction operation to the state $Q_i$: 
If $\langle a \rightarrow \beta \bullet c \delta, k \rangle \in Q_i$, then
the parser tries to recognize $c\delta$ after having read `the first i input tokens`.  To this end
it has to parse $c$ in the string `the input suffix starting at i`.
Therefore, if $c \rightarrow \gamma$ is a rule of our grammar,
we add the *Earley item* $\langle c \rightarrow \bullet \gamma, i\rangle$ to the set $Q_i$:
$$ \langle a \rightarrow \beta \bullet c \delta, k\rangle \in Q_i 
       \wedge (c \rightarrow \gamma) \in R 
       \;\rightarrow\;
       Q_i := Q_i \cup\bigl\{ \langle c \rightarrow \bullet\gamma, i\rangle\bigr\}.
$$
As the right hand side $\gamma$ might start with a variable, the function uses a fix point iteration
until no more *Earley items* are added to $Q_i$.

The function `scan tokens chart i` applies the scanning operation to the state $Q_i$.

If $\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i$ and $T$ is a token,
then the parser tries to recognize the right hand side of the grammar rule
$$ a \rightarrow \beta T \gamma$$ 
and after having read `input tokens from k to i, with i excluded` it has already recognized  $\beta$.
If we now have `tokens[i].Kind = T`, then the parser still has to recognize $\gamma$ in `the input suffix starting at i + 1`.
Therefore, the *Earley object* $\langle a \rightarrow \beta T \bullet \gamma, k\rangle$ is added to
the set $Q_{i+1}$:
$$\langle a \rightarrow \beta \bullet T \gamma, k\rangle \in Q_i \wedge x_{i+1} = T
       \;\rightarrow\;
       Q_{i+1} := Q_{i+1} \cup \bigl\{ \langle a \rightarrow \beta T \bullet \gamma, k\rangle \bigr\}
$$

`tokenize` retains both a token kind and its lexeme, so the recognizer and tree builder can share the scanner.

In [ ]:
open System
open System.Text.RegularExpressions

let tokenize (source: string) =
    let scanner = Regex(@"\G(?:\s+|[1-9][0-9]*|0|[a-zA-Z][a-zA-Z0-9_]*|[-+*/()])")
    let rec loop position result =
        if position = source.Length then List.rev result
        else
            let token = scanner.Match(source, position)
            if not token.Success then failwithf "Illegal character at column %d" (position + 1)
            let next = position + token.Length
            if String.IsNullOrWhiteSpace token.Value then loop next result
            else
                let kind =
                    if Char.IsDigit token.Value[0] then "NUMBER"
                    elif Char.IsLetter token.Value[0] then "IDENTIFIER"
                    else token.Value
                loop next ({ Kind = kind; Lexeme = token.Value } :: result)
    loop 0 []

In [ ]:
printfn "%A" (tokenize "1 + 2 * 3")

The function `test` takes two arguments.
- `file` is the name of a file containing a grammar,
- `word` is a string that should be parsed.

`word` is first tokenized.  Then the resulting token list is parsed using *Earley's algorithm*.

In [ ]:
let test file word =
    let grammar = parse file
    let tokens = tokenize word
    let chart = earleyParse grammar tokens
    chart |> Array.iteri (fun position items -> printfn "Q%d = %A" position items)
    let accepted = Set.contains (finishItem grammar) chart[tokens.Length]
    printfn "Accepted: %b" accepted
    accepted

In [ ]:
checkNotebook (test "simple.g" "1 + 2 * 3")
checkNotebook (not (recognizes (parse "simple.g") (tokenize "1 + * 2")))
// Nullable productions require closure at the same input position.
let nullable = Parse.text "s: a a; a: NUMBER; a: ;"
checkNotebook (recognizes nullable [])
checkNotebook (recognizes nullable (tokenize "7"))
checkNotebook (recognizes nullable (tokenize "7 7"))